<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def preparar_dados(estado_aleatorio=42):
    dataset = fetch_openml(
        name='mnist_784',
        version=1,
        as_frame=False,
        return_X_y=True
    )
    
    imagens, rotulos = dataset[0], dataset[1].astype(int)

    proporcao_teste = 0.2
    entradas_treino, entradas_teste, saidas_treino, saidas_teste = train_test_split(
        imagens,
        rotulos,
        test_size=proporcao_teste,
        random_state=estado_aleatorio,
        stratify=rotulos
    )
    
    return entradas_treino, entradas_teste, saidas_treino, saidas_teste

# Verificação dos dados carregados
entradas_treino, entradas_teste, saidas_treino, saidas_teste = preparar_dados(estado_aleatorio=42)

print("Formato X_train :", entradas_treino.shape)
print("Formato X_test  :", entradas_teste.shape)
print("Formato y_train :", saidas_treino.shape)
print("Formato y_test  :", saidas_teste.shape)

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def criar_floresta_aleatoria(features_treino, classes_treino, semente=42):
    classificador = RandomForestClassifier(
        random_state=semente,
        n_estimators=100
    )
    classificador.fit(features_treino, classes_treino)
    return classificador


def criar_adaboost(features_treino, classes_treino, semente=42):
    classificador = AdaBoostClassifier(
        random_state=semente,
        n_estimators=100
    )
    classificador.fit(features_treino, classes_treino)
    return classificador

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score

def calcular_desempenho(classificador, dados_teste, gabarito):
    predicoes = classificador.predict(dados_teste)

    acuracia = accuracy_score(gabarito, predicoes)

    return acuracia

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def executar_pipeline(tipo_classificador="rf", semente=42):
    features_treino, features_teste, classes_treino, classes_teste = carregar_dados(semente)

    if tipo_classificador == "rf":
        classificador = criar_floresta_aleatoria(features_treino, classes_treino, semente)
    elif tipo_classificador == "ab":
        classificador = criar_adaboost(features_treino, classes_treino, semente)
    else:
        raise ValueError("tipo_classificador deve ser 'rf' ou 'ab'")

    acuracia = calcular_desempenho(classificador, features_teste, classes_teste)

    return acuracia

executar_pipeline()

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

niveis_profundidade = [1, 3, 5, 10, 15, 20, None]

print(f"{'max_depth':<12} {'Treino':>8} {'Teste':>8}")
print("-" * 30)
for nivel in niveis_profundidade:
    arvore = DecisionTreeClassifier(random_state=42, max_depth=nivel)
    arvore.fit(features_treino, classes_treino)
    desempenho_treino = accuracy_score(classes_treino, arvore.predict(features_treino))
    desempenho_teste  = accuracy_score(classes_teste,  arvore.predict(features_teste))
    print(f"{str(nivel):<12} {desempenho_treino:>8.4f} {desempenho_teste:>8.4f}")

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acuracia_ada = executar_pipeline("ab")
acuracia_rf  = executar_pipeline("rf")

print("Acurácia AdaBoost:", acuracia_ada)
print("Acurácia Random Forest:", acuracia_rf)

features_treino, features_teste, classes_treino, classes_teste = carregar_dados(semente=42)

classificador_rf  = criar_floresta_aleatoria(features_treino, classes_treino, semente=42)
classificador_ada = criar_adaboost(features_treino, classes_treino, semente=42)

print(f"\n{'Métrica':<12} {'Random Forest':>15} {'AdaBoost':>12}")
print("-" * 42)

for titulo, clf in [("Random Forest", classificador_rf), ("AdaBoost", classificador_ada)]:
    predicoes = clf.predict(features_teste)
    print(f"\n{titulo}")
    print(f"  Accuracy  : {accuracy_score(classes_teste, predicoes):.4f}")
    print(f"  Precision : {precision_score(classes_teste, predicoes, average='weighted'):.4f}")
    print(f"  Recall    : {recall_score(classes_teste, predicoes, average='weighted'):.4f}")
    print(f"  F1-Score  : {f1_score(classes_teste, predicoes, average='weighted'):.4f}")

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
acuracia_ada  = executar_pipeline("ab")
acuracia_rf   = executar_pipeline("rf")

print("Acurácia AdaBoost com Seed=42:", acuracia_ada)
print("Acurácia Random Forest com Seed=42:", acuracia_rf)

acuracia_ada  = executar_pipeline("ab", 7)
acuracia_rf   = executar_pipeline("rf", 7)

print("Acurácia AdaBoost com Seed=7:", acuracia_ada)
print("Acurácia Random Forest com Seed=7:", acuracia_rf)

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
desempenho_treino = accuracy_score(classes_treino, classificador_rf.predict(features_treino))
desempenho_teste  = accuracy_score(classes_teste,  classificador_rf.predict(features_teste))

print("Acurácia no Treino:", desempenho_treino)
print("Acurácia no Teste :", desempenho_teste)
print("Diferença :", round(desempenho_treino - desempenho_teste, 4))

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
qtd_estimadores = [10, 50, 100, 200]

print(f"{'n_estimators':<15} {'Random Forest':>15} {'AdaBoost':>12}")
print("-" * 44)

for qtd in qtd_estimadores:
    floresta = RandomForestClassifier(random_state=42, n_estimators=qtd)
    floresta.fit(features_treino, classes_treino)
    resultado_rf = accuracy_score(classes_teste, floresta.predict(features_teste))

    boost = AdaBoostClassifier(random_state=42, n_estimators=qtd)
    boost.fit(features_treino, classes_treino)
    resultado_ada = accuracy_score(classes_teste, boost.predict(features_teste))

    print(f"{qtd:<15} {resultado_rf:>15.4f} {resultado_ada:>12.4f}")

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [ ]:
#Aqui estão as respostas reescritas:
#1. A acurácia, por si só, não é uma métrica suficiente para avaliar modelos de forma completa. Apesar de ser intuitiva e fácil de interpretar, ela pode mascarar falhas importantes, principalmente em problemas com várias classes ou distribuições desequilibradas. O uso conjunto de métricas como precisão, recall e F1-score enriquece a análise, pois permite examinar o desempenho do modelo em cada classe individualmente e detectar tipos de erro que passariam despercebidos olhando apenas para a acurácia.
#2. Para assegurar que os resultados obtidos não sejam fruto do acaso, é fundamental controlar a aleatoriedade através do parâmetro random_state e realizar os experimentos com múltiplas seeds diferentes. Isso permite verificar se os resultados se mantêm consistentes entre execuções e diminui a influência de uma divisão específica dos dados, tornando as conclusões estatisticamente mais sólidas.
#3. Dois problemas metodológicos que podem comprometer este experimento são: o uso de uma única divisão treino/teste, que tende a introduzir viés nas estimativas de desempenho, e a falta de otimização dos hiperparâmetros, que pode tornar a comparação entre os modelos injusta. Acrescenta-se ainda a limitação de avaliar os modelos com uma única métrica, o que restringe a profundidade da análise.
#4. O pipeline desenvolvido apresenta um nível razoável de confiabilidade, dado que possui estrutura modular e garante reprodutibilidade pelo uso de seeds fixas. Contudo, sua robustez é comprometida por depender de uma única partição dos dados e por não contemplar técnicas como validação cruzada ou busca por hiperparâmetros ideais, recursos que seriam essenciais para uma avaliação verdadeiramente confiável.